# LinguaAI: Multilingual Neural Machine Translation Experiments
### NLLB-200 Pretrained Model Evaluation & Many-to-Many Translation

This notebook demonstrates direct many-to-many translation between Indic and global languages using the Meta AI NLLB-200 Transformer.

In [ ]:
import sys
sys.path.append('..')
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model_name = 'facebook/nllb-200-distilled-600M'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()
print('NLLB-200 Model loaded successfully!')

## 1. Direct Multilingual Translation: Hindi -> Tamil

In [ ]:
hindi_text = 'नमस्ते, आप कैसे हैं?'
tokenizer.src_lang = 'hin_Deva'
inputs = tokenizer(hindi_text, return_tensors='pt').to(device)

with torch.no_grad():
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids('tam_Taml'),
        num_beams=2
    )

tamil_output = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
print('Hindi Source: ', hindi_text)
print('Tamil Output: ', tamil_output)

## 2. Quantitative Evaluation: SacreBLEU & chrF++

In [ ]:
from evaluation.metrics import compute_sacrebleu, compute_chrf

hypotheses = [tamil_output]
references = ['வணக்கம், எப்படி இருக்கிறீர்கள்?']

bleu = compute_sacrebleu(hypotheses, references)
chrf = compute_chrf(hypotheses, references)
print(f'SacreBLEU Score: {bleu:.2f}')
print(f'chrF++ Score:   {chrf:.2f}')